# Lab 07: Comparing Classification Algorithms

What's today?

See different algorithms on different types of data, and observe what are the properties of each one of this models

In [118]:
import sys 
sys.path.append("../")
from utils import *

In [125]:
import numpy as np
import pandas as pd

from IPython.display import display, Math, Latex


import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import PolynomialFeatures

In [120]:
def plot_all_classifiers(Zs):
    fig = make_subplots(rows=2, cols=3,
    subplot_titles=("Logistic regression","Desicion Tree", "KNN", "SVM", "SVM with polynomial kernel", "SVM with rbf kernel"))\
    
    for i, z in enumerate(Zs):
        fig.add_trace(go.Contour(x=xrange, y=yrange, z=z, colorscale='RdBu', reversescale=True, opacity=0.8), 
                      row=(i//3) + 1, col=(i%3)+1)
    fig.update_layout(height=600, width=800, title_text="Side By Side Subplots")
    fig.show()


In [121]:
X = np.random.uniform(low=-5, high=5, size=(1200, 2))

grid_step = 0.2

# Z is the probability grid to create the desicion bounderies
x_min, x_max = np.min(X[:, 0]) - grid_step, np.max(X[:, 0]) + grid_step
y_min, y_max = np.min(X[:, 1]) - grid_step, np.max(X[:, 1]) + grid_step

xrange = np.arange(x_min, x_max, grid_step)
yrange = np.arange(y_min, y_max, grid_step)
xx, yy = np.meshgrid(xrange, yrange)

# Models 

Models are:


1. Logistic regression

    $$ P \left(y = 1\right) = \sigma \left(w^T \cdot x\right)   \\   w, x \in R^{d+1} $$


2. Desicion Tree

    $$ y = \sum_{i=1}^n C_i \mathbf{1}_{B_{i}} $$ 
    
    $ B_j $ is the i'th 'box' in surface that the classifier partitioned. \
    $ C_j $ is the classification of the i'th box. \
    n number of boxes
    

3. K Nearest Neighbors
    
    $$ \underset{y\in \{0,1\}}{argmax} \sum_{i=1}^k \mathbf{1}_{y_{\pi_{i}} = y}  $$ 

4. SVM
    
    $$ y = sign \left(w^T \cdot \phi\left(x\right)\right)   \\   w, x \in R^{m} $$
    
    for w and $$ \phi(\cdot) $$ that minimize the objective. By the representer theorem:
    
    $$ y = sign \left(\sum_{i=1}^m \alpha_i \cdot K\left(x_i, x \right) \right) $$
    
    Where K is the defined as  $$ K\left(x_i, x_j \right) = <\phi(x_i), \phi(x_j)> $$
    
    1. SVM with linear kernel
   
    $$ K\left(x_i, x_j \right) = x_i^T \cdot x_j $$

    2. SVM with Polynomial kernel
    
    $$ K\left(x_i, x_j \right) = \left(1 + \left(x_i^T \cdot x_j\right)\right)^2 $$
    
    3. SVM with gaussian kernel
    
    $$ K\left(x_i, x_j \right) = exp\left(-\frac{1}{2}|| x_i - x_j || \right) $$



In [122]:
## Gaussian for rbf, gamma=1/2

models = [
    LogisticRegression(),
    DecisionTreeClassifier(max_depth=5), 
    KNeighborsClassifier(n_neighbors=5),
    SVC(kernel='linear'),
    SVC(kernel='poly', degree=2, gamma=1.0, coef0=1), 
    SVC(gamma=0.5)
]


# Generating Data

## Linear data

In [123]:
def even_classes(X, y):
    counts = [np.sum(y == i) for i in [0, 1]]
    min_counts = np.min(counts)
    
    final_X = []
    final_y = []
    for c in [0, 1]:
        final_X.append(X[y == c][:min_counts])
        final_y.append(y[y == c][:min_counts])
    
    final_X = np.array(final_X) 
    final_y = np.array(final_y)

    final_X = final_X.reshape((final_X.shape[0]*final_X.shape[1], final_X.shape[2])) 
    final_y = final_y.reshape((final_y.shape[0]*final_y.shape[1]))

    random_perm = np.random.permutation(final_X.shape[0])
    final_X, final_y = final_X[random_perm], final_y[random_perm]
    return final_X, final_y



def oracle_classifier_linear(X):
    labels = []
    for i in range(X.shape[0]):
        labels.append(1 if (X[i][0]) < (X[i][1]) else 0)
    return np.array(labels)
        
        
y_linear = oracle_classifier_linear(X)
X, y_linear = even_classes(X, y_linear)

go.Figure([go.Scatter(x = X[:, 0], y=X[:, 1], mode = 'markers',
        marker = dict(color=y_linear, colorscale='Portland'))]).show()

In [124]:
Zs = []

for model in models:
    model.fit(X, y_linear)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    Zs.append(Z)

plot_all_classifiers(Zs)

## Circular data

In [110]:
#Oracle is the real data classifier - it has the true rule of the classification
def oracle_classifier_circle(X):
    labels = []
    for i in range(X.shape[0]):
        labels.append(1 if (X[i][0])**2  + (X[i][1])**2 <= 3**2 else 0)
    return np.array(labels)
        
        
y_circular = oracle_classifier_circle(X)
X, y_circular = even_classes(X, y_circular)


go.Figure([go.Scatter(x = X[:, 0], y=X[:, 1], mode = 'markers',
        marker = dict(color=y_circular, colorscale='Portland'))]).show()

In [111]:
Zs = []

for model in models:
    model.fit(X, y_circular)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    Zs.append(Z)

plot_all_classifiers(Zs)

In [112]:
def oracle_classifier_tree(X):
    labels = []
    for i in range(X.shape[0]):
        labels.append(1 if ((-5 <= X[i][0] <= 0 and -4 <= X[i][1] <= 0) 
                            or (1 <= X[i][0] <= 5 and -1 <= X[i][1] <= 4)) else 0)
    return np.array(labels)

y_tree = oracle_classifier_tree(X)
X, y_tree = even_classes(X, y_tree)


go.Figure([go.Scatter(x = X[:, 0], y=X[:, 1], mode = 'markers',
        marker = dict(color=y_tree, colorscale='Portland'))]).show()

In [113]:
Zs = []

for model in models:
    model.fit(X, y_tree)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    Zs.append(Z)

plot_all_classifiers(Zs)

Notice:
    
1. Logistic regression and SVM create linear bounderies.

2. Desicion trees make parallel to the axes bouneries.

3. Note how the SVM bouneries are able to reach more complex bounderies as the kernel is more complex 
But, the SVM is basically a linear classifier. That is an importent concept - using mappings and different representations of the data to is also a tool to make the hypothesis class richer.


